# Class 4 — The Data Lakehouse: S3, Delta Lake, and the Medallion Architecture (EMR)

## 1. Why raw files on S3 aren't enough

A data lake is "just files in object storage" (S3, ADLS, GCS). That's cheap and infinitely scalable, but plain Parquet/CSV/JSON files on S3 give you none of the guarantees a database gives you:

- **No atomicity** -- if a job writing 500 Parquet files crashes after writing 300, readers can see a half-written, inconsistent dataset.
- **No isolation** -- a reader running concurrently with a writer can see a mix of old and new files.
- **No schema enforcement** -- nothing stops a bad job from writing a column with the wrong type or an extra column, silently corrupting downstream consumers.
- **No update/delete** -- object storage is append/overwrite-oriented; there's no efficient way to update or delete a subset of rows without rewriting entire files/partitions.
- **No versioning/audit trail** -- you can't easily ask "what did this table look like an hour ago."

The **lakehouse** pattern layers a **table format** (a transaction log/metadata layer) on top of plain files in object storage, giving you database-like guarantees while keeping data lake economics. Delta Lake, Apache Iceberg, and Apache Hudi are the three major table formats; this project uses **Delta Lake**.

## 2. How Delta Lake actually works

A Delta table is: (a) a directory of Parquet data files, plus (b) a `_delta_log/` directory of JSON (and periodic Parquet checkpoint) files describing every change ever made to the table.

```text
s3://bucket/tables/silver_clickstream/
├── _delta_log/
│   ├── 00000000000000000000.json   <- version 0: CREATE TABLE + initial files
│   ├── 00000000000000000001.json   <- version 1: added files from an append
│   ├── 00000000000000000002.json   <- version 2: a MERGE (some files removed, new files added)
│   └── ...
├── part-00000-....snappy.parquet
├── part-00001-....snappy.parquet
└── ...
```

Each log entry is a set of **actions**: `add` a file, `remove` a file, change metadata, etc. A reader determines "what does this table look like right now" by replaying the log forward -- this is what gives you **snapshot isolation**.

This design is what enables: **ACID transactions**, **time travel** (`VERSION AS OF`/`TIMESTAMP AS OF`), **schema enforcement/evolution**, **MERGE (upsert)** as one atomic transaction, and **streaming source + sink from the same table**.

## 3. The medallion architecture (bronze / silver / gold)

```text
  Kafka/MSK,          BRONZE                 SILVER                  GOLD
  batch files    -->  raw, append-only, -->  cleaned, deduped,  -->  business-level
                      schema-on-write,       validated,             aggregates,
                      minimal transform      enriched with          dimensional models,
                                             dimensions             ready for BI/ML
```

- **Bronze** -- as close to the source as possible, plus ingestion metadata. Kept append-only so you can always reprocess silver/gold from bronze.
- **Silver** -- validated, deduplicated, type-cast, enriched with reference data.
- **Gold** -- aggregated, business-metric tables shaped for a specific consumption pattern.

## 4. Delta Lake operational patterns you'll use constantly

- `MERGE INTO` -- the standard upsert pattern for CDC ingestion, slowly-changing dimensions, and streaming `foreachBatch` writes to gold tables (see `02_spark_structured_streaming.ipynb`, step 4).
- `OPTIMIZE table ZORDER BY (col1, col2)` -- compacts small files and co-locates rows with similar values so range/filter queries skip more data.
- `VACUUM table` -- physically deletes data files no longer referenced by the log and older than the retention threshold (default 7 days).
- `DESCRIBE HISTORY table` -- every version, its operation type, and metrics. Your audit trail.

## 5. Delta vs Iceberg (why you'll hear both names)

| | Delta Lake | Apache Iceberg |
|---|---|---|
| Native engine | Originated at Databricks (also OSS, multi-engine via Delta Standalone/UniForm) | Multi-engine by design (Spark, Trino, Flink, Snowflake, etc.) |
| Metadata layout | `_delta_log` JSON commits + periodic Parquet checkpoints | metadata.json + manifest lists + manifests |
| When to choose | Deepest feature/performance integration on this project's stack (EMR + open-source `delta-spark`) | Strong multi-engine interoperability requirement |

They solve the *same* underlying problem with different metadata designs. This project defaults to Delta everywhere, and demonstrates a managed Iceberg table for comparison in `emr-notebooks/04_delta_production_patterns_iceberg.ipynb`.

Below, a compact runnable walkthrough of the concepts above.

Run the cell below first -- it configures Delta Lake for this notebook's Spark session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "class_emr"
base_path = "s3://<your-lakehouse-bucket>/data/class-emr"

def table(name):
    return f"`{schema}`.`{name}`"

def path(*parts):
    return base_path.rstrip("/") + "/" + "/".join(p.strip("/") for p in parts)

from pyspark.sql import functions as F

spark.sql(f"CREATE DATABASE IF NOT EXISTS `{schema}` LOCATION '{path('tables')}'")
spark.sql(f"USE `{schema}`")

## ACID + schema enforcement

In [ ]:
dim_customer = spark.createDataFrame(
    [("u00001", "loyal", "west"), ("u00002", "new", "east")],
    "user_id string, segment string, region string",
)
dim_customer.write.mode("overwrite").format("delta").saveAsTable(table("class_lakehouse_dim_customer"))

try:
    bad_schema = spark.createDataFrame([("u00003", 42)], "user_id string, segment int")  # wrong type for segment
    bad_schema.write.mode("append").format("delta").saveAsTable(table("class_lakehouse_dim_customer"))
except Exception as e:
    print("Rejected as expected -- Delta enforces schema on write:")
    print(type(e).__name__, str(e)[:300])

## MERGE (upsert) — the core dimensional/CDC pattern

Using plain SQL `MERGE INTO` against a temp view (no `delta.tables.DeltaTable` Python API needed -- that package isn't installed on this project's EMR clusters).

In [ ]:
updates = spark.createDataFrame(
    [("u00001", "loyal", "central"), ("u00003", "active", "south")],
    "user_id string, segment string, region string",
)
updates.createOrReplaceTempView("customer_updates")

spark.sql(f"""
MERGE INTO {table("class_lakehouse_dim_customer")} AS t
USING customer_updates AS s
ON t.user_id = s.user_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
spark.table(table("class_lakehouse_dim_customer")).toPandas()

## Time travel

In [ ]:
history = spark.sql(f"DESCRIBE HISTORY {table('class_lakehouse_dim_customer')}")
earliest_version = history.select(F.min("version")).first()[0]
print(f"Table as of version {earliest_version} (before the MERGE):")
spark.read.option("versionAsOf", earliest_version).table(table("class_lakehouse_dim_customer")).toPandas()

## OPTIMIZE and file layout

In [ ]:
try:
    spark.sql(f"OPTIMIZE {table('class_lakehouse_dim_customer')} ZORDER BY (region)")
except Exception as e:
    print("OPTIMIZE/ZORDER not supported by this delta-spark version:", str(e)[:200])

spark.sql(f"DESCRIBE DETAIL {table('class_lakehouse_dim_customer')}").toPandas()

## Discussion / what's next

You've now seen every concept the production pipeline uses:

- Batch DataFrame processing and Spark's execution model (class 1).
- Structured Streaming, watermarks, and stateful aggregation (class 2).
- Kafka/MSK as the durable ingestion log and its schema-parsing contract with Spark (class 3).
- Delta Lake as the transactional table format that makes bronze/silver/gold reliable (class 4).

Move to `RUNBOOK.md` and `emr-notebooks/00_environment_setup.ipynb` to build the real, end-to-end retail clickstream pipeline: S3 + MSK + EMR + Delta, with production Python packaging and MWAA orchestration.